# ARENA3 런타임 C — GPT 학습기 G1

선택 지점끼리만 잇는 **SMDP 듀얼링 DQN**(M4에서 출발, 모드 6개, 관측 119칸)을 학습 난수 42·43·44로 학습합니다.
난수마다 20 CPU-시간(합 60)에서 멈추고, 끝나면 심판 코드로 84상황 × 모델 3개를 평가합니다.

1. 런타임을 **CPU**로 고르고 **런타임 → 모두 실행**을 누르세요.
2. 순서: **실행 전 점검**(진짜 SUMO로 한 묶음을 따로 폴더에, 약 5분) → 학습 3개 동시에(약 10시간) → 심판 평가 252회(약 1시간) → 판정.
3. 끊기면 다시 **모두 실행**을 누르세요. 학습은 마지막 체크포인트에서 이어집니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3.zip"
A3 = DRIVE / "arena3"
WORKSPACE = Path("/content/FutureTraffic_ARENA3")
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", A3)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_V1", "ARENA3 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(재현 점검이 실패할 수 있어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 실행 전 점검: 진짜 SUMO로 한 묶음(에피소드 2개)을 점검용 폴더에 학습합니다. 본 학습 폴더와 섞이지 않습니다.
SMOKE = A3 / "G1_smoke"
if (SMOKE / "seed_42" / "checkpoint.json").exists():
    print("이전 점검 기록이 있어 건너뜁니다:", colab.read_json(SMOKE / "seed_42" / "checkpoint.json"))
else:
    colab.run_live(["src/arena3_g1.py", "train", "--seed", "42", "--out", str(SMOKE), "--work", "/content/g1_smoke",
                    "--max-batches", "1"], WORKSPACE)

In [ ]:
# 2) 학습: 난수 42·43·44를 동시에(각각 에피소드 2개씩 병렬). 셀을 멈춰도 학습은 계속 돕니다. 5분마다 진행을 보여 줍니다.
TRAIN = A3 / "G1_train"
LOGS = A3 / "logs" / "C"
jobs = {f"g1_seed{s}": ["src/arena3_g1.py", "train", "--seed", str(s), "--out", str(TRAIN), "--work", f"/content/g1_{s}"]
        for s in (42, 43, 44)}
pids = colab.launch(jobs, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300)
assert code == 0, "G1 학습에서 오류가 났습니다. 위 로그를 저에게 보내 주세요."

In [ ]:
# 3) 심판 평가: 모델 3개 × 84상황 = 252회, 그다음 판정(런타임 A의 기준선이 있어야 판정합니다).
finals = [TRAIN / f"seed_{s}" / "final" for s in (42, 43, 44)]
assert all((f / "model.pt").exists() for f in finals), "학습이 아직 끝나지 않았습니다. 2)를 다시 실행하세요."
colab.run_live(["src/arena3_referee.py", "learner", "--name", "G1", "--models", *map(str, finals),
                "--out", str(A3 / "referee" / "G1"), "--work", "/content/g1_eval", "--workers", str(CPUS)], WORKSPACE)
BASE = A3 / "referee" / "baseline"
colab.run_live(["src/arena3_referee.py", "judge", "--baseline", str(BASE), "--candidate", f"W4={BASE}",
                "--candidate", f"BPRIME={A3 / 'referee' / 'bprime'}", "--candidate", f"G1={A3 / 'referee' / 'G1'}",
                "--out", str(A3 / "referee")], WORKSPACE)
summary = A3 / "referee" / "summary.md"
if summary.exists():
    display(Markdown(summary.read_text()))